# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Anotación celular

**Flujo de trabajo:**

1. Cargar objeto con clusters + diagnóstico de calidad (salida de 05b) y archivo de anotación oficial
2. Alinear barcodes entre ambos objetos
3. Transferir anotación oficial → adata.obs
4. Validar coherencia entre clusters Leiden y anotación oficial
5. Visualizar UMAP con tipos celulares anotados
6. Calcular marcadores diferenciales por cluster (DEG) para validación
7. Guardar objeto anotado

Se mantiene el cálculo de marcadores diferenciales (DEG) para verificar que los top genes por cluster concuerdan con las anotaciones.



# · Importaciones y configuración



In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:15
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from collections import Counter
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import yaml
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder

sc.settings.verbosity = 1


# · Configuración de rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

DIAG_H5AD_DIR = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['06_cluster_diagnostics']}"
INPUT_PATH    = f"{DIAG_H5AD_DIR}/{PARAMS['outputs']['clustered_diagnostics_h5ad']}"

DIAG_TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables']['06_cluster_diagnostics']}"

ANNO_PATH   = f"{PROJECT_ROOT}/{PARAMS['paths']['raw']['annotation_file']}"
OUTPUT_DIR  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['07_annotated']}"
OUTPUT_PATH = f"{OUTPUT_DIR}/{PARAMS['outputs']['annotated_h5ad']}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables']['07_annotation']}"
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures']['07_annotation']}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR

BATCH_KEY     = "sample_id"
CONDITION_KEY = "condition"
MAIN_KEY      = "leiden_r0.8"
N_PCS         = 20
PALETTE       = {'HC': '#2E86AB', 'UC': '#E84855', 'CD': '#F4A261'}

# Umbral de pureza por debajo del cual no se asigna el tipo oficial mayoritario a ciegas
PURITY_MIN = 0.4

print("Configuración cargada")

Configuración cargada


In [ ]:
adata = sc.read_h5ad(INPUT_PATH)
df_anno = pd.read_csv(ANNO_PATH, compression='gzip')

sample_barcodes = df_anno['cell_id'].astype(str).head(10).tolist()
print(f"\nEjemplo de barcodes en el archivo: {sample_barcodes[:3]}")
print(f"Ejemplo de barcodes en adata.obs_names: {adata.obs_names[:3]}")

# Limpiar barcodes: quedarse con la parte después del último '_'
adata.obs['barcode_raw'] = adata.obs_names.str.split('_').str[-1]

# Clave única = muestra + barcode, calculada en ambos lados desde aquí
adata.obs['match_key'] = (
    adata.obs['sample_id'].astype(str).str.split('_').str[-1]   # "GSM6614348_HC-1" -> "HC-1"
    .str.replace('-', '', regex=False)                          # "HC-1" -> "HC1"
    + '_' + adata.obs['barcode_raw']
)

df_anno['match_key'] = (
    df_anno['sample'].astype(str) + '_' + df_anno['cell_id'].astype(str)
)


assert adata.obs['match_key'].is_unique, "Hay match_key duplicadas."
print(f"   {adata.obs[MAIN_KEY].nunique()} clusters detectados")

# En el diagnóstico de 06 se estableció la identidad de linaje por marcadores canónicos.
# Se usa más abajo para no asignar un tipo mayoritario poco fiable.

df_marker_diag = pd.read_csv(f"{DIAG_TABLES_DIR}/diag_marker_identity.csv")
best_lineage_map  = dict(zip(df_marker_diag['cluster'].astype(str), df_marker_diag['best_lineage_match']))
weak_identity_map = dict(zip(df_marker_diag['cluster'].astype(str), df_marker_diag['flag_weak_identity']))
print(f"   Diagnóstico de marcadores (06) cargado: {len(df_marker_diag)} clusters")



Ejemplo de barcodes en el archivo: ['AAACCTGCAAGTCTGT-1', 'AAACCTGGTTATGCGT-1', 'AAACCTGTCGGCATCG-1']
Ejemplo de barcodes en adata.obs_names: Index(['GSM6614348_HC-1_AAACCTGAGGTCGGAT-1',
       'GSM6614348_HC-1_AAACCTGCAAGTCTGT-1',
       'GSM6614348_HC-1_AAACCTGGTCGTCTTC-1'],
      dtype='object')
   21 clusters detectados
   Diagnóstico de marcadores (06) cargado: 21 clusters


# · Inspección del archivo de anotación

In [ ]:
# Archivo anotado cargado antes
print(f"Dimensiones: {df_anno.shape[0]} filas, {df_anno.shape[1]} columnas")
print(f"Columnas: {df_anno.columns.tolist()}")
print(f"Primeras filas:\n{df_anno.head(2)}")

# Mostrar información de tipos de datos y valores nulos
print("\nInformación de tipos y nulos:")
print(df_anno.info())

# Verificar formato de los barcodes
sample_barcodes = df_anno['cell_id'].astype(str).head(10).tolist()
print(f"\nEjemplo de barcodes en el archivo: {sample_barcodes[:3]}")
print(f"Ejemplo de barcodes en adata.obs_names: {adata.obs_names[:3]}")

# Verificar barcodes
barcodes_adata = set(adata.obs['match_key'])
barcodes_anno  = set(df_anno['match_key'])
common  = barcodes_adata.intersection(barcodes_anno)
missing = barcodes_adata - barcodes_anno
extra   = barcodes_anno - barcodes_adata

print(f"\n--- CRUCE DE BARCODES ---")
print(f"Células en adata: {len(barcodes_adata)}")
print(f"Células en anotación: {len(barcodes_anno)}")
print(f"Células comunes: {len(common)} ({100*len(common)/len(barcodes_adata):.1f}% de adata)")
print(f"Células sin anotación: {len(missing)}")
print(f"Células anotadas no presentes en adata (filtradas por QC): {len(extra)}")

if len(common) < 0.7 * len(barcodes_adata):
    print(" Baja cobertura. Verificar formato de barcodes.")
else:
    print(" Cobertura suficiente. Procediendo con merge.")

# Tipos celulares presentes
print("\nTipos celulares (top 10):")
print(df_anno['annotation'].value_counts().head(10))


Dimensiones: 46702 filas, 6 columnas
Columnas: ['Unnamed: 0', 'sample', 'cell_id', 'annotation', 'nanostring_reference', 'match_key']
Primeras filas:
                  Unnamed: 0 sample             cell_id annotation  \
0  SC_002_AAACCTGCAAGTCTGT-1    HC1  AAACCTGCAAGTCTGT-1  CD4 naïve   
1  SC_002_AAACCTGGTTATGCGT-1    HC1  AAACCTGGTTATGCGT-1     gd IEL   

  nanostring_reference               match_key  
0                  CD4  HC1_AAACCTGCAAGTCTGT-1  
1               gd IEL  HC1_AAACCTGGTTATGCGT-1  

Información de tipos y nulos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46702 entries, 0 to 46701
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Unnamed: 0            46702 non-null  object
 1   sample                46702 non-null  object
 2   cell_id               46702 non-null  object
 3   annotation            46702 non-null  object
 4   nanostring_reference  46702 non-null  object
 5 

# 🔧 Anotación oficial y merge

In [ ]:
df_anno_idx = (df_anno[df_anno['match_key'].isin(common)]
               .copy()
               .drop_duplicates(subset='match_key')
               .set_index('match_key'))

adata.obs['cell_type_official'] = (
    df_anno_idx['annotation'].reindex(adata.obs['match_key']).values
)
adata.obs['cell_type_official'] = adata.obs['cell_type_official'].fillna('Unknown')

# Columna de categoría amplia
# 'nanostring_reference' es la categoría coarse usada en el paper original
if 'nanostring_reference' in df_anno_idx.columns:
    adata.obs['cell_type_coarse'] = (
        df_anno_idx['nanostring_reference'].reindex(adata.obs['match_key']).fillna('Unknown').values
    )
    print("    Columna 'cell_type_coarse' desde 'nanostring_reference'")
else:
    adata.obs['cell_type_coarse'] = adata.obs['cell_type_official']
    print("     'nanostring_reference' no encontrada — usando anotación fina")

# Resumen
n_annotated = (adata.obs['cell_type_official'] != 'Unknown').sum()
print(f"\n   Células con anotación oficial: {n_annotated:,} "
      f"({100*n_annotated/adata.n_obs:.1f}%)")
print(f"   Tipos celulares únicos: {adata.obs['cell_type_official'].nunique()}")



    Columna 'cell_type_coarse' desde 'nanostring_reference'

   Células con anotación oficial: 38,213 (88.1%)
   Tipos celulares únicos: 92


# · Silhouette de condición calculado por tipo celular.

Permite comprobar si la señal HC/UC/CD se conserva dentro de cada linaje aunque el silhouette global de condición sea negativo.

In [ ]:
def condition_asw_by_celltype(X, obs, condition_key='condition',
                               celltype_key='cell_type_coarse',
                               min_cells=50, min_conditions=2,
                               sample_size=2000, random_state=42):
    rows = []
    for ct in obs[celltype_key].unique():
        if ct == 'Unknown':          # <-- única línea añadida
          continue
        mask = (obs[celltype_key].values == ct)
        n_cells = mask.sum()
        if n_cells < min_cells:
            continue
        conds_present = obs.loc[mask, condition_key].unique()
        if len(conds_present) < min_conditions:
            continue

        le = LabelEncoder()
        cond_enc = le.fit_transform(obs.loc[mask, condition_key])
        n_sample = min(sample_size, n_cells)

        sil = silhouette_score(X[mask], cond_enc,
                                sample_size=n_sample,
                                random_state=random_state)
        rows.append({
            'cell_type': ct,
            'n_cells': int(n_cells),
            'n_conditions': len(conds_present),
            'silhouette_raw': sil,
            'asw_norm': (sil + 1) / 2,
        })
    return pd.DataFrame(rows).sort_values('n_cells', ascending=False)

# ── Ejecutar sobre embeddings pre y post Harmony ──────────────────────
N_PCS = 20
df_pre  = condition_asw_by_celltype(adata.obsm['X_pca'][:, :N_PCS], adata.obs)
df_post = condition_asw_by_celltype(adata.obsm['X_pca_harmony'],    adata.obs)

comparison = df_pre.merge(
    df_post, on='cell_type', suffixes=('_pre', '_post')
)
comparison['delta_asw'] = comparison['asw_norm_post'] - comparison['asw_norm_pre']
print(comparison[['cell_type', 'n_cells_pre', 'asw_norm_pre',
                   'asw_norm_post', 'delta_asw']].to_string(index=False))

                   cell_type  n_cells_pre  asw_norm_pre  asw_norm_post  delta_asw
                         CD4         4038      0.508769       0.491648  -0.017121
                      PC IgA         3992      0.527901       0.497387  -0.030514
                 Colonocytes         3883      0.539565       0.492619  -0.046946
                      PC IgG         3024      0.545818       0.528661  -0.017157
                         CD8         2882      0.501400       0.494077  -0.007324
            Epithelium Ribhi         1693      0.525856       0.491801  -0.034055
                      Goblet         1613      0.514194       0.502322  -0.011872
           PC IgA heat shock         1479      0.559720       0.512924  -0.046796
                      B cell         1339      0.486127       0.475857  -0.010270
               T cells CCL20         1159      0.512158       0.499210  -0.012948
                       Tregs         1103      0.495434       0.490367  -0.005067
                

### - Comprobación de robustez del linaje destacado

In [ ]:
def bootstrap_silhouette(X_sub, cond_enc, n_boot=200, random_state=42):
    rng = np.random.default_rng(random_state)
    scores = []
    for _ in range(n_boot):
        idx = rng.choice(len(cond_enc), size=len(cond_enc), replace=True)
        if len(set(cond_enc[idx])) < 2:
            continue
        scores.append(silhouette_score(X_sub[idx], cond_enc[idx]))
    return np.array(scores)

ct = 'Inflammatory fibroblasts'
mask = adata.obs['cell_type_coarse'].values == ct
le = LabelEncoder()
cond_enc = le.fit_transform(adata.obs.loc[mask, 'condition'])

boot_pre  = bootstrap_silhouette(adata.obsm['X_pca'][mask, :20], cond_enc)
boot_post = bootstrap_silhouette(adata.obsm['X_pca_harmony'][mask], cond_enc)

print(f"Pre-Harmony:  media={boot_pre.mean():.3f}  IC95%=[{np.percentile(boot_pre,2.5):.3f}, {np.percentile(boot_pre,97.5):.3f}]")
print(f"Post-Harmony: media={boot_post.mean():.3f}  IC95%=[{np.percentile(boot_post,2.5):.3f}, {np.percentile(boot_post,97.5):.3f}]")

Pre-Harmony:  media=0.308  IC95%=[0.238, 0.386]
Post-Harmony: media=0.103  IC95%=[0.048, 0.149]


### - Contrastar con marcadores biológicos

In [ ]:
markers_inflam_fibro = ['IL11', 'TNFRSF11B', 'MMP3', 'CHI3L1']
sc.pl.violin(
    adata[mask], markers_inflam_fibro,
    groupby='condition', rotation=45,
    save='_inflammatory_fibroblasts_by_condition.png'
)

### - Comprobar distribución de la población entre clusters

In [ ]:
dist_cluster = (
    adata.obs.loc[mask, MAIN_KEY]          # antes: 'leiden'
    .value_counts(normalize=True)
    .mul(100).round(1)
)
print(dist_cluster.head(5))

leiden_r0.8
7    100.0
1      0.0
2      0.0
3      0.0
0      0.0
Name: proportion, dtype: float64


# · Validación entre clustering y anotación


In [ ]:
print(f"\n→ Validando coherencia clusters vs anotación oficial...")
cluster_rows = []
for cl in sorted(adata.obs[MAIN_KEY].unique(), key=lambda x: int(x)):
    sub = adata.obs[adata.obs[MAIN_KEY] == cl]
    # Excluir 'Unknown' del cálculo de pureza
    sub_known = sub[sub['cell_type_coarse'] != 'Unknown']
    if len(sub_known) == 0:
        major_type, purity = 'Unknown', 0.0
    else:
        vc = sub_known['cell_type_coarse'].value_counts()
        major_type = vc.index[0]
        purity     = vc.iloc[0] / len(sub)
    cl_str    = str(cl)
    marker_ok = not weak_identity_map.get(cl_str, True)
    if purity < PURITY_MIN:
        consensus_type = (f"{best_lineage_map.get(cl_str, major_type)} (subtipo indeterminado)"
                           if marker_ok else "Ambiguous (sin identidad clara)")
    else:
        consensus_type = major_type

    cluster_rows.append({
        'cluster':              cl,
        'major_type':           major_type,
        'purity':               round(purity, 3),
        'n_cells':              len(sub),
        'best_lineage_markers': best_lineage_map.get(cl_str, 'NA'),
        'consensus_type':       consensus_type
    })
    flag = "⚠️" if purity < PURITY_MIN else "✅"
    print(f"   C{int(cl):>2}: oficial={major_type:<20} → consenso={consensus_type:<32} "
          f"purity={purity:.2f} n={len(sub):,} {flag}")

df_purity = pd.DataFrame(cluster_rows)
df_purity.to_csv(f"{TABLES_DIR}/cluster_to_major_celltype.csv", index=False)

low_purity = df_purity[df_purity['purity'] < PURITY_MIN]['cluster'].tolist()
if low_purity:
    print(f"\n     Clusters con pureza < {PURITY_MIN}: {low_purity}")

# Tipo consenso final por cluster: ya incorpora pureza + evidencia de marcadores
cluster_to_consensus = dict(zip(df_purity['cluster'], df_purity['consensus_type']))
adata.obs['cell_type_consensus'] = (
    adata.obs[MAIN_KEY].map(cluster_to_consensus).fillna('Unknown')
)



→ Validando coherencia clusters vs anotación oficial...
   C 0: oficial=CD8                  → consenso=CD8                              purity=0.54 n=4,865 ✅
   C 1: oficial=Colonocytes          → consenso=Ambiguous (sin identidad clara)  purity=0.40 n=4,673 ⚠️
   C 2: oficial=CD4                  → consenso=CD4                              purity=0.53 n=4,436 ✅
   C 3: oficial=PC IgA               → consenso=PC IgA                           purity=0.47 n=4,162 ✅
   C 4: oficial=PC IgA               → consenso=PC IgA                           purity=0.49 n=3,675 ✅
   C 5: oficial=CD4                  → consenso=CD4                              purity=0.45 n=3,451 ✅
   C 6: oficial=PC IgG               → consenso=PC IgG                           purity=0.78 n=2,939 ✅
   C 7: oficial=S1                   → consenso=Fibroblast (subtipo indeterminado) purity=0.35 n=2,478 ⚠️
   C 8: oficial=Colonocytes          → consenso=Colonocytes                      purity=0.81 n=2,421 ✅
   C 9: ofic

## · Validación de clusters de baja pureza: dobletes y estabilidad de partición

Para los clusters que no alcanzan el umbral de pureza y cuyo consenso resulta ambiguo, se comprueba si el origen es técnico (dobletes) o de resolución (sobre-partición de una población real), antes de decidir cómo tratarlos en el resto del pipeline.

In [ ]:
low_purity_ambiguous = df_purity.loc[
    df_purity['consensus_type'].str.startswith('Ambiguous') | (df_purity['purity'] <= 0.05),
    'cluster'
].tolist()

doublet_check, partition_check = {}, {}
for cl in low_purity_ambiguous:
    mask = adata.obs[MAIN_KEY] == str(cl)
    doublet_check[cl] = adata.obs.loc[mask, 'doublet_score'].mean()
    partition_check[cl] = (adata.obs.loc[mask, 'leiden_r0.5']
                            .value_counts(normalize=True))

global_doublet = adata.obs['doublet_score'].mean()
for cl in low_purity_ambiguous:
    top_parent, top_frac = partition_check[cl].index[0], partition_check[cl].iloc[0]
    print(f"C{cl}: doublet={doublet_check[cl]:.3f} (global={global_doublet:.3f}) | "
          f"{top_frac*100:.1f}% en r0.5-cluster {top_parent}")

C1: doublet=0.057 (global=0.054) | 95.9% en r0.5-cluster 3
C15: doublet=0.054 (global=0.054) | 77.0% en r0.5-cluster 13
C19: doublet=0.167 (global=0.054) | 96.5% en r0.5-cluster 4
C20: doublet=0.243 (global=0.054) | 75.7% en r0.5-cluster 6


Con la tasa de dobletes y la estabilidad de partición ya calculadas, se resuelve cada cluster ambiguo según su causa identificada, generando la etiqueta definitiva (cell_type_final) que usarán los notebooks posteriores

In [ ]:
DOUBLET_THRESHOLD = 2.5  # múltiplo de la tasa global; documentado como umbral orientativo
PARENT_CONCENTRATION_MIN = 0.90  # concentración mínima en r0.5 para heredar etiqueta

adata.obs['cell_type_final'] = adata.obs['cell_type_consensus'].astype(str)

for cl in low_purity_ambiguous:
    mask = adata.obs[MAIN_KEY] == str(cl)
    is_doublet = doublet_check[cl] >= DOUBLET_THRESHOLD * global_doublet
    top_parent, top_frac = partition_check[cl].index[0], partition_check[cl].iloc[0]
    is_stable_partition = top_frac >= PARENT_CONCENTRATION_MIN

    if is_doublet and is_stable_partition:
        parent_mask = adata.obs['leiden_r0.5'] == top_parent
        parent_type = adata.obs.loc[parent_mask, 'cell_type_official'].value_counts().index[0]
        adata.obs.loc[mask, 'cell_type_final'] = f"{parent_type} (heredado de r0.5-C{top_parent}, doublet score elevado)"
    elif is_doublet:
        adata.obs.loc[mask, 'cell_type_final'] = 'Doublet'
    elif is_stable_partition:
        parent_mask = adata.obs['leiden_r0.5'] == top_parent
        parent_type = adata.obs.loc[parent_mask, 'cell_type_official'].value_counts().index[0]
        adata.obs.loc[mask, 'cell_type_final'] = f"{parent_type} (heredado de r0.5-C{top_parent})"
    else:
        adata.obs.loc[mask, 'cell_type_final'] = 'Unresolved'

vc = adata.obs['cell_type_final'].value_counts()
print(vc[vc.index.str.contains('Doublet|heredado|Unresolved')])

cell_type_final
Epithelium Ribhi (heredado de r0.5-C3)                   4673
Unresolved                                                505
PC IgG 1 (heredado de r0.5-C4, doublet score elevado)     115
Doublet                                                   111
Name: count, dtype: int64


# · Visualización de la anotación oficial

In [ ]:
print("\n→ Generando figuras de anotación...")
umap = adata.obsm['X_umap_harmony']

# ── FIGURA 1: UMAP con anotación oficial fina ─────────────────────────────────

freq_fine       = adata.obs['cell_type_official'].value_counts()
types_to_keep   = freq_fine[freq_fine >= 50].index.tolist()
adata.obs['cell_type_official_plot'] = adata.obs['cell_type_official'].apply(
    lambda x: x if x in types_to_keep else 'Other'
)

fig, ax = plt.subplots(figsize=(14, 12))
unique_types = sorted(adata.obs['cell_type_official_plot'].unique())
cmap_fine    = plt.get_cmap('tab20', len(unique_types))
for i, t in enumerate(unique_types):
    mask = adata.obs['cell_type_official_plot'] == t
    ax.scatter(umap[mask, 0], umap[mask, 1],
               s=0.5, alpha=0.4, color=cmap_fine(i),
               label=t, rasterized=True)
ax.set_title(
    "UMAP con anotación oficial del estudio \n",
    fontsize=12
)
n_cols_legend = min(10, len(unique_types))
ax.legend(
    markerscale=6, fontsize=7,
    ncol=n_cols_legend,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.08),
    frameon=False,
    columnspacing=0.8,
    handletextpad=0.3
)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/umap_official_annotation.png",
            dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: umap_official_annotation.png")

# ── FIGURA 2: UMAP con categoría coarse ──────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 11))
coarse_types   = sorted(adata.obs['cell_type_coarse'].unique())
cmap_coarse    = plt.get_cmap('Set2', len(coarse_types))
for i, t in enumerate(coarse_types):
    mask = adata.obs['cell_type_coarse'] == t
    ax.scatter(umap[mask, 0], umap[mask, 1],
               s=0.5, alpha=0.4, color=cmap_coarse(i),
               label=t, rasterized=True)
ax.set_title("UMAP con categorías amplias \n", fontsize=12)
n_cols_coarse = min(7, len(coarse_types))
ax.legend(
    markerscale=6, fontsize=9,
    ncol=n_cols_coarse,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.08),
    frameon=False,
    columnspacing=0.8,
    handletextpad=0.3
)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/umap_coarse_annotation.png",
            dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: umap_coarse_annotation.png")

# ── FIGURA 3: Matriz de confusión cluster vs tipo celular coarse ──────────────
# Muestra qué proporción de cada cluster corresponde a cada tipo celular.

contingency      = pd.crosstab(adata.obs[MAIN_KEY], adata.obs['cell_type_coarse'])
contingency_norm = contingency.div(contingency.sum(axis=1), axis=0)

fig_h = max(8, contingency_norm.shape[0] * 0.4)
fig_w = max(14, contingency_norm.shape[1] * 0.5)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
sns.heatmap(
    contingency_norm, annot=True, fmt='.2f', cmap='Blues',
    annot_kws={'size': 7}, linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'Proporción en cluster'},
    ax=ax
)
ax.set_title(
    "Homogeneidad cluster vs tipo celular \n",
    fontsize=12
)
ax.set_xlabel("Tipo celular (coarse)", fontsize=10)
ax.set_ylabel("Cluster (Leiden)", fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/cluster_vs_celltype_confusion.png",
            dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: cluster_vs_celltype_confusion.png")

# ── FIGURA 4: Pureza de clusters ──────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, max(5, len(df_purity) * 0.4)))
colors_p = ['#E84855' if p < 0.4 else '#F4A261' if p < 0.6 else '#2E86AB'
            for p in df_purity['purity']]
bars = ax.barh(
    [f"C{int(r.cluster)}: {r.major_type}" for _, r in df_purity.iterrows()],
    df_purity['purity'],
    color=colors_p, edgecolor='none'
)
ax.axvline(0.4, color='red',    ls='--', lw=1.2, label='Límite bajo (0.4)')
ax.axvline(0.7, color='green',  ls='--', lw=1.2, label='Bueno (0.7)')
ax.set_xlabel('Pureza del cluster ', fontsize=10)
ax.set_title('Pureza de clusters Leiden vs anotación oficial \n', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/cluster_purity.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Guardada: cluster_purity.png")



→ Generando figuras de anotación...
   → Guardada: umap_official_annotation.png
   → Guardada: umap_coarse_annotation.png
   → Guardada: cluster_vs_celltype_confusion.png
   → Guardada: cluster_purity.png


# · Marcadores diferenciales (DEG) por cluster

- Método Wilcoxon: no paramétrico, robusto para distribuciones asimétricas típicas de scRNA-seq con muchos ceros.


In [ ]:
print(f"\n→ Calculando marcadores diferenciales por cluster ")
    #  f"(Wilcoxon, use_raw={use_raw})...")

sc.tl.rank_genes_groups(
    adata,
    groupby=MAIN_KEY,
    method='wilcoxon',
    use_raw= False,
    n_genes=50,
    key_added='rank_genes_harmony'
)
print("   Marcadores calculados")

# Guardar top 10 genes por cluster en CSV
top_genes_list = []
for cl in sorted(adata.obs[MAIN_KEY].unique(), key=int):
    df_g = sc.get.rank_genes_groups_df(
        adata, group=cl, key='rank_genes_harmony'
    )
    df_g = df_g.head(10)
    df_g['cluster']    = cl
    df_g['major_type'] = cluster_to_consensus.get(cl, 'Unknown')
    top_genes_list.append(df_g)

top_genes_df = pd.concat(top_genes_list, ignore_index=True)
top_genes_df.to_csv(
    f"{TABLES_DIR}/top10_markers_per_cluster_{MAIN_KEY}.csv",
    index=False, float_format='%.4f'
)
print(f"   → CSV guardado: top10_markers_per_cluster_{MAIN_KEY}.csv")

# Heatmap de top 3 genes por cluster para la visualización de la firma transcriptómica
sc.pl.rank_genes_groups_heatmap(
    adata,
    n_genes=3,
    groupby=MAIN_KEY,
    key='rank_genes_harmony',
    use_raw=False,
    show_gene_labels=True,
    figsize=(14, 12),
    dendrogram=True,
    standard_scale='var',       # normalizar cada gen a [0,1] para comparabilidad
    show=False,
    save=f"_top3_markers_heatmap_{MAIN_KEY}.png"
)
print(f"   → Heatmap guardado")


→ Calculando marcadores diferenciales por cluster 
   Marcadores calculados
   → CSV guardado: top10_markers_per_cluster_leiden_r0.8.csv
   → Heatmap guardado


# · Figuras de marcadores canónicos

In [ ]:
# Dot plot de marcadores canónicos por cluster
markers_canonical = {
    "Epitelial":   ["EPCAM", "KRT20", "MUC2"],
    "Goblet":      ["MUC2", "TFF3"],
    "T cell":      ["CD3D", "CD3E", "CD8A", "CD4"],
    "B cell":      ["CD79A", "MS4A1"],
    "Plasma":      ["IGHA1", "MZB1"],
    "Macrophage":  ["C1QA", "CD68"],
    "Monocyte":    ["LYZ", "S100A8"],
    "Fibroblast":  ["COL1A1", "DCN"],
    "Endothelial": ["VWF", "PECAM1"],
    "Mast":        ["TPSAB1", "CPA3"],
    "Tuft":        ["POU2F3", "TRPM5"]
}
all_markers = list(dict.fromkeys([
    g for g_list in markers_canonical.values()
    for g in g_list if g in adata.var_names
]))
if all_markers:
    sc.pl.dotplot(
        adata,
        var_names=all_markers,
        groupby=MAIN_KEY,
        standard_scale='var',
        show=False,
        save=f"_canonical_markers_dotplot_{MAIN_KEY}.png"
    )
    print(f"   → Dot plot de marcadores guardado")

   → Dot plot de marcadores guardado


# · Columna de anotación concenso

In [ ]:
# El consenso ya se calculó en la celda de validación de pureza, aquí solo se resume.

print("\n   Tipos consenso por cluster:")
print(adata.obs['cell_type_consensus'].value_counts().to_string())



   Tipos consenso por cluster (pureza + marcadores canónicos de 06):
cell_type_consensus
CD4                                   7887
PC IgA                                7837
Ambiguous (sin identidad clara)       5178
CD8                                   4865
PC IgG                                2939
Fibroblast (subtipo indeterminado)    2478
Colonocytes                           2421
B cell                                2226
Goblet                                2187
Monocyte (subtipo indeterminado)      2132
Mast                                   874
Plasma (subtipo indeterminado)         778
N1                                     554
Tuft cells                             435
Endothelium                            304
Glia                                   182
Goblet (subtipo indeterminado)         111


# · Guardar objeto anotado

In [ ]:
print(f"\n→ Guardando objeto anotado en {OUTPUT_PATH} ...")
adata.write_h5ad(OUTPUT_PATH, compression='gzip')


→ Guardando objeto anotado en /content/drive/MyDrive/IBD_TFM/data/interim/07_annotated/IBD_annotated.h5ad ...
